In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (10).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (3).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (9).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (5).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (7).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (6).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (2).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (4).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/IMG_20260110_131939925.jpg (8).jpeg
/kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos/root_phenotyping_results.

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.morphology import skeletonize
from scipy.spatial import ConvexHull

# ==========================================================
# 1. CONFIGURATION & CALIBRATION
# ==========================================================
# Update this if your scale changes (derived from the ruler)
SCALE_FACTOR = 0.066  # mm per pixel

# This hunts for your "cropped_photos" folder anywhere in the Kaggle Input
input_dir = Path("/kaggle/input/datasets/arnavitis/cropped-photos-2")
folders = [f for f in input_dir.rglob("cropped_photos") if f.is_dir()]

if not folders:
    print("⚠️ Error: Could not find 'cropped_photos' folder. Please ensure the dataset is attached.")
else:
    SOURCE_DIR = folders[0]
    print(f"✅ Source Directory Set: {SOURCE_DIR}")

    all_results = []

    # ==========================================================
    # 2. BATCH PROCESSING ENGINE
    # ==========================================================
    image_files = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"🚀 Processing {len(image_files)} images...")

    for filename in image_files:
        img_path = os.path.join(SOURCE_DIR, filename)
        
        # Load in Grayscale for Otsu Math
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if raw_img is None: continue

        # --- A. OTSU SEGMENTATION ---
        # Gaussian Blur reduces mat texture noise
        blurred = cv2.GaussianBlur(raw_img, (7, 7), 0)
        # Otsu calculates threshold by minimizing intra-class variance
        _, binary_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        # Morphological Opening removes small dust pixels
        kernel = np.ones((3,3), np.uint8)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

        # --- B. ARCHITECTURAL METRICS ---
        points = np.column_stack(np.where(binary_mask > 0))
        
        if len(points) > 3:
            # Metric 1: Skeletonization (Total Root Length)
            # Recursive thinning to a 1-pixel centerline
            skeleton = skeletonize(binary_mask > 0)
            trl_px = np.sum(skeleton)
            trl_mm = trl_px * SCALE_FACTOR
            
            # Metric 2: Convex Hull (Spatial Exploration Area)
            # Smallest convex polygon encompassing all pixels
            hull = ConvexHull(points)
            area_px2 = hull.volume 
            area_mm2 = area_px2 * (SCALE_FACTOR ** 2)
            
            # Metric 3: System Tortuosity
            # Ratio of actual length to vertical depth
            max_y, min_y = np.max(points[:, 0]), np.min(points[:, 0])
            depth_px = max_y - min_y
            depth_mm = depth_px * SCALE_FACTOR
            tortuosity = trl_px / depth_px if depth_px > 0 else 0
            
            all_results.append({
                "Image_Name": filename,
                "TRL_mm": round(trl_mm, 2),
                "Depth_mm": round(depth_mm, 2),
                "Tortuosity": round(tortuosity, 3),
                "Hull_Area_mm2": round(area_mm2, 2)
            })

    # ==========================================================
    # 3. EXPORT & FINAL PREVIEW
    # ==========================================================
    df = pd.DataFrame(all_results)
    df.to_csv("final_phenotyping_report.csv", index=False)
    
    print("\n" + "="*40)
    print("📊 BATCH ANALYSIS COMPLETE")
    print("="*40)
    print(f"Average Batch Tortuosity: {df['Tortuosity'].mean():.2f}")
    print(f"Average Root Length: {df['TRL_mm'].mean():.2f} mm")
    print("="*40)
    
    # Display the full 10-plant table
    display(df)

✅ Source Directory Set: /kaggle/input/datasets/arnavitis/cropped-photos-2/cropped_photos
🚀 Processing 10 images...

📊 BATCH ANALYSIS COMPLETE
Average Batch Tortuosity: 2.17
Average Root Length: 105.03 mm


,Image_Name,TRL_mm,Depth_mm,Tortuosity,Hull_Area_mm2
0,IMG_20260110_131939925.jpg (10).jpeg,135.10,41.65,3.244,638.56
1,IMG_20260110_131939925.jpg (3).jpeg,78.74,62.83,1.253,1059.38
2,IMG_20260110_131939925.jpg (9).jpeg,124.21,34.19,3.633,343.29
3,IMG_20260110_131939925.jpg (5).jpeg,127.84,59.80,2.138,832.55
4,IMG_20260110_131939925.jpg (7).jpeg,95.50,52.87,1.806,581.75
5,IMG_20260110_131939925.jpg (6).jpeg,199.91,57.35,3.486,1316.86
6,IMG_20260110_131939925.jpg (2).jpeg,65.41,50.75,1.289,600.73
7,IMG_20260110_131939925.jpg (4).jpeg,83.36,63.36,1.316,1269.78
8,IMG_20260110_131939925.jpg (8).jpeg,89.89,36.43,2.467,337.96
9,IMG_20260110_131939925.jpg (1).jpeg,50.36,47.78,1.054,251.35
